In [10]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# 1. Team Configuration & Path Anchoring
project_root = Path(r"C:\Users\DELL\dsba6171-microcosm-starter")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env", override=True)

from langchain_community.document_loaders import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from src.vectorstore.chain import get_rag_chain

# 2. Domain Data & Canonical Representation
csv_path = project_root / "data" / "structured" / "raw" / "synthetic_knowledge_items.csv"
chroma_dir = project_root / "data" / "processed" / "chroma"

# Load documents and preserve metadata
loader = CSVLoader(
    file_path=str(csv_path),
    metadata_columns=["ki_id", "jurisdiction", "authority_status", "effective_date"]
)
documents = loader.load()

# 3. Embed & Load into Chroma
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=str(chroma_dir)
)

# 4. Run Baseline Semantic Retrieval
query = "What is the policy for ChromaDB vector database storage?"
baseline_results = vectorstore.similarity_search(query, k=2)

print("=== 1. BASELINE SEMANTIC RETRIEVAL (Unfiltered Top-2) ===")
for i, doc in enumerate(baseline_results, 1):
    print(f"Result {i}:")
    print(f"  ID: {doc.metadata.get('ki_id')} | Jurisdiction: {doc.metadata.get('jurisdiction')} | Status: {doc.metadata.get('authority_status')}")
    print(f"  Content: {doc.page_content[:100]}...\n")

# 5. Run Metadata/Context-Filtered Retrieval
filtered_results = vectorstore.similarity_search(
    query,
    k=2,
    filter={"$and": [{"jurisdiction": "US"}, {"authority_status": "active"}]}
)

print("=== 2. CONTEXT-FILTERED RETRIEVAL (Jurisdiction: US, Status: active) ===")
for i, doc in enumerate(filtered_results, 1):
    print(f"Result {i}:")
    print(f"  ID: {doc.metadata.get('ki_id')} | Jurisdiction: {doc.metadata.get('jurisdiction')} | Status: {doc.metadata.get('authority_status')}")
    print(f"  Content: {doc.page_content[:100]}...\n")

# 6. Compare Results (Hit@K Evaluation)
target_ki_id = "KI-001"
baseline_hit = any(doc.metadata.get("ki_id") == target_ki_id for doc in baseline_results)
filtered_hit = any(doc.metadata.get("ki_id") == target_ki_id for doc in filtered_results)

print("=== 3. RETRIEVAL EVALUATION (Hit@1 for Active US Policy KI-001) ===")
print(f"Target KI-001 present in Baseline Top-2: {baseline_hit}")
print(f"Target KI-001 present in Filtered Top-2: {filtered_hit}\n")

# 7. Trace Evidence to Source
print("=== 4. SOURCE EVIDENCE TRACE ===")
top_filtered_doc = filtered_results[0]
print(f"Selected Document ID: {top_filtered_doc.metadata.get('ki_id')}")
print(f"Source File: {top_filtered_doc.metadata.get('source')}")
print(f"Row Number: {top_filtered_doc.metadata.get('row')}")
print(f"Effective Date: {top_filtered_doc.metadata.get('effective_date')}\n")

# 8. Optional: Generate Grounded AI Response
print("=== 5. OPTIONAL: GROUNDED AI RESPONSE ===")
chain = get_rag_chain()
response = chain.invoke(query)
print("LLM Output:\n", response)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== 1. BASELINE SEMANTIC RETRIEVAL (Unfiltered Top-2) ===
Result 1:
  ID: KI-004 | Jurisdiction: US | Status: superseded
  Content: ki_topic: Vector Database Policy
ki_text: Legacy ChromaDB persistence rules apply to archival databa...

Result 2:
  ID: KI-001 | Jurisdiction: US | Status: active
  Content: ki_topic: Vector Database Policy
ki_text: ChromaDB is an open-source vector database designed to sto...

=== 2. CONTEXT-FILTERED RETRIEVAL (Jurisdiction: US, Status: active) ===
Result 1:
  ID: KI-001 | Jurisdiction: US | Status: active
  Content: ki_topic: Vector Database Policy
ki_text: ChromaDB is an open-source vector database designed to sto...

=== 3. RETRIEVAL EVALUATION (Hit@1 for Active US Policy KI-001) ===
Target KI-001 present in Baseline Top-2: True
Target KI-001 present in Filtered Top-2: True

=== 4. SOURCE EVIDENCE TRACE ===
Selected Document ID: KI-001
Source File: C:\Users\DELL\dsba6171-microcosm-starter\data\structured\raw\synthetic_knowledge_items.csv
Row Number: 0

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM Output:
 The policy for ChromaDB vector database storage is to store high-dimensional vector embeddings locally.
